# Rehavision デモ Notebook

人工知能システム開発 第7班「Rehavision」

Google Colab上でGemini APIを使い、Prompt Template + Reference（RAG風の外部情報）に基づいて、
作業療法士・家族からの質問にAIが回答し、TTSで読み上げるデモです。

課題PDF「デモ」スライドで示されている、複数モデルへのフォールバック方式（クォータ超過・廃止モデル対策）を実装しています。

## 使い方
1. Colabメニューの鍵アイコン（Secrets）から `GOOGLE_API_KEY` を登録し、Notebook access をONにする
2. 上から順にセルを実行する
3. 「質問」セルの `question` を書き換えて再実行すると、任意の質問で試せる

## 1. セットアップ

In [ ]:
!pip install -q google-generativeai gTTS

In [ ]:
import os
import google.generativeai as genai
from gtts import gTTS
from IPython.display import Audio, display

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except ImportError:
    api_key = os.environ.get("GOOGLE_API_KEY")

assert api_key, "GOOGLE_API_KEY が見つかりません。Colab Secretsまたは環境変数に設定してください。"
genai.configure(api_key=api_key)
print("Gemini API 設定完了")

## 2. モデルのフォールバック呼び出し
無料枠のクォータ超過（429）やモデル廃止（404）に対応するため、候補モデルを順に試す。

In [ ]:
MODEL_CANDIDATES = [
    "models/gemini-2.5-flash",
    "models/gemini-2.5-pro",
    "models/gemini-2.0-flash",
    "models/gemini-2.0-flash-001",
    "models/gemini-2.0-flash-lite-001",
    "models/gemini-2.0-flash-lite",
    "models/gemini-flash-latest",
]


def ask_custom_llm(prompt, model_candidates=MODEL_CANDIDATES, verbose=True):
    last_error = None
    for model_name in model_candidates:
        if verbose:
            print(f"-> {model_name} で実行を試みます...")
        try:
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(prompt)
            if verbose:
                print(f"\u2705 成功 ({model_name})")
            return response.text.strip()
        except Exception as exc:
            last_error = exc
            if verbose:
                print(f"[失敗] {model_name}: {exc}")
            continue
    raise RuntimeError(f"すべての候補モデルで失敗しました: {last_error}")

## 2.5 使えるモデルを確認する（本番前に必ず実行）

APIキーによって使えるモデルが違い、廃止（404）やクォータ超過（429）も起きる。
下のセルで実際に呼べるモデルを特定し、`MODEL_CANDIDATES` の先頭に持ってくること。

In [ ]:
# APIキーで参照できるモデルを列挙
available = [m.name for m in genai.list_models()
             if "generateContent" in m.supported_generation_methods]
print(f"参照可能なモデル：{len(available)}件")
for name in available:
    print(" ", name)

# 実際に1回呼んでみて、本当に応答が返るモデルだけを残す
print("\n=== 実際に呼び出せるか確認 ===")
usable = []
for name in [m for m in MODEL_CANDIDATES if m in available] or available[:8]:
    try:
        genai.GenerativeModel(name).generate_content("こんにちは")
        print(f"\u2705 {name}")
        usable.append(name)
    except Exception as exc:
        print(f"\u274c {name}: {str(exc)[:80]}")

if usable:
    MODEL_CANDIDATES = usable + [m for m in MODEL_CANDIDATES if m not in usable]
    print(f"\n\u2705 使えるモデル {len(usable)}件を候補の先頭に設定しました：{usable[0]}")
else:
    print("\n\u26a0\ufe0f 呼び出せるモデルがありません。APIキーとクォータを確認してください。")

## 3. Prompt Template / Reference の読み込み

リポジトリ（`rehavision/prompts/07_prompt.txt`, `07_reference.txt`）をColabにアップロードまたはgit cloneしている場合は
ファイルから読み込む。単体で使う場合は、下のインラインの文字列にフォールバックする。

In [ ]:
PROMPT_TEMPLATE_PATH = "../prompts/07_prompt.txt"
REFERENCE_PATH = "../prompts/07_reference.txt"

INLINE_PROMPT_TEMPLATE = """# 回答条件
- あなたは「Rehavision」の音声アシスタントです。作業療法士または患者のご家族からの質問に、下記「患者情報」に基づいて答えてください。
- 医療専門用語を使う場合は、家族にも分かる簡単な言葉で補足してください。
- 「患者情報」に記載がない内容は、推測で断定せず「記録には該当する情報がありません。担当の作業療法士にご確認ください。」と回答してください。
- 診断や治療方針の変更につながるような断定的な発言は避けてください。
- 回答は3文以内、日本語の自然な話し言葉でまとめてください。

# フォーマット
- 箇条書きや記号(・, -, *, #)は使わないでください。音声合成(TTS)でそのまま読み上げるため、話し言葉の文章のみを出力してください。
- 文末は「です」「ます」調で統一してください。

# 患者情報
{reference_str}

# 質問
質問：{question}
"""

INLINE_REFERENCE = """■ 患者基本情報
氏名：田中 一郎
主病名：右大腿骨頸部骨折 術後
現在の病期：回復期

■ 現在の身体状態
歩行時に軽度のふらつきがあり、単独歩行は転倒リスクが高いため、歩行時は見守りが必要です。
バランス能力には改善傾向が見られます。
更衣動作は自立に近い状態です。

■ 訓練進捗
理学療法は全10回のうち8回を実施済みで、達成度は80%です。
理学療法の開始日は〇月〇日です。
実施した訓練：歩行訓練、バランス訓練、立ち上がり訓練
前回の訓練：立ち上がり訓練と歩行訓練を実施
本日のメニュー：歩行訓練とバランス訓練

■ 訓練結果（数値）
歩行距離：10m → 45m に改善
片脚立位：2秒 → 8秒 に改善

■ 課題
長時間の連続歩行が苦手です。
方向転換時の重心移動に課題があります。

■ 今後の方針・見通し
次回は歩行の安定性向上を目的とした訓練を予定しています。
今後も安定性を重視した訓練を継続する方針です。
退院目標は、安全に自力で歩行できるようになることです。
現状を維持できれば、約〇週間で目標達成の見込みです。

■ 退院後の注意点
自宅内での転倒予防として、移動時の照明、敷物、段差に注意する必要があります。
"""

try:
    with open(PROMPT_TEMPLATE_PATH, encoding="utf-8") as f:
        prompt_template = f.read()
    with open(REFERENCE_PATH, encoding="utf-8") as f:
        reference_str = f.read()
    print("prompts/ ディレクトリから読み込みました")
except FileNotFoundError:
    prompt_template = INLINE_PROMPT_TEMPLATE
    reference_str = INLINE_REFERENCE
    print("ファイルが見つからないため、インラインのテンプレートを使用します")

## 4. 質問応答デモ
審査員役の質問をここに入力して実行する。

In [ ]:
question = "リハビリの効果は出ている？"

prompt = prompt_template.format(reference_str=reference_str, question=question)
print("=== LLMの動作テスト ===")
print(f"ユーザー：{question}")
answer = ask_custom_llm(prompt)
print(f"AIの回答：{answer}\n")

## 5. TTSで読み上げ

In [ ]:
tts = gTTS(text=answer, lang="ja")
tts.save("answer.mp3")
display(Audio("answer.mp3", autoplay=False))

## 6. 複数質問での一括テスト（WOZコーパス想定）
課題資料のWOZ対話例（田中さんのバイタル表示、コーヒーの可否など）を想定した質問リストで動作確認する。

In [ ]:
test_questions = [
    # 07_testset.md の20問から代表的なものを抜粋
    "患者の状態はどうですか？",
    "必要なサポートは？",
    "リハビリの効果は出ている？",
    "訓練はどこまで進んでいますか",
    "患者さんの苦手なことは？",
    "作業療法士の記録を確認して",  # 条件B（RAGなし）で記録を捏造した質問。RAGありで防げるか確認
]

print("=== LLMの動作テスト ===")
for i, q in enumerate(test_questions):
    print(f"ユーザー：{q}")
    a = ask_custom_llm(prompt_template.format(reference_str=reference_str, question=q), verbose=False)
    print(f"AIの回答：{a}\n")